# IDSW v2.0.0 Workflow: Bayesian Models, Bayesian Neural Networks & Hyperparameter Tuning

This notebook demonstrates the new **IDSW 2.0.0** capabilities in:
- `idsw.modelling.bayes`: Gaussian, Multinomial, and Bernoulli Naive Bayes, Bayesian Linear Regression (Ridge/ARD), and Bayesian Logistic Regression.
- `idsw.modelling.deep`: Bayesian Neural Networks (`get_bayesian_tf_model`, `get_siamese_bayesian_model`) with Monte Carlo Dropout uncertainty quantification (`predict_with_uncertainty`).
- `idsw.modelling.tuning`: Grid Search, Random Search, Coarse-to-Fine Informed Search, Bayesian Optimization (GP-UCB), and Genetic Algorithm tuning, with interactive `plotly=True` visualizations.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification, make_regression
from sklearn.model_selection import train_test_split

from idsw import *
ControlVars.show_plots = True
ControlVars.show_results = True

## 1. Bayesian Classification & Regression (`idsw.modelling.bayes`)

In [ ]:
# Generate synthetic classification dataset
X_clf, y_clf = make_classification(n_samples=300, n_features=6, random_state=42)
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_clf, y_clf, test_size=0.25, random_state=42)

# Naive Bayes Classifier
nb_model, nb_metrics = naive_bayes_classifier(
    X_train=X_train_c,
    y_train=y_train_c,
    variant='gaussian',
    X_test=X_test_c,
    y_test=y_test_c
)

# Bayesian Logistic Regression
blog_model, blog_metrics = bayesian_logistic_regression(
    X_train=X_train_c,
    y_train=y_train_c,
    X_test=X_test_c,
    y_test=y_test_c
)

In [ ]:
# Generate synthetic regression dataset
X_reg, y_reg = make_regression(n_samples=250, n_features=5, noise=5.0, random_state=42)
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X_reg, y_reg, test_size=0.25, random_state=42)

# Bayesian Linear Regression with predictive uncertainty (std)
blin_model, blin_metrics, y_pred_mean, y_pred_std = bayesian_linear_regression(
    X_train=X_train_r,
    y_train=y_train_r,
    method='ridge',
    X_test=X_test_r,
    y_test=y_test_r
)
print(f"First 5 test predictions +/- 1 std: {list(zip(y_pred_mean[:5].round(2), y_pred_std[:5].round(2)))}")

## 2. Hyperparameter Tuning (`idsw.modelling.tuning`) with Plotly

In [ ]:
from sklearn.ensemble import RandomForestClassifier
base_rf = RandomForestClassifier(random_state=42)

# 1. Grid Search Tuning with Plotly
best_grid_model, best_grid_params, grid_df = grid_search_tuning(
    model_object=base_rf,
    param_grid={'n_estimators': [20, 50], 'max_depth': [3, 5, None]},
    X_train=X_train_c,
    y_train=y_train_c,
    cv=3,
    plotly=True,
    export_png=True,
    file_name='grid_search_tuning_demo'
)

# 2. Bayesian Optimization Tuning with Plotly
best_bo_model, best_bo_params, bo_history = bayesian_optimization_tuning(
    model_object=base_rf,
    param_space={'n_estimators': (10, 100), 'max_depth': (2, 12)},
    X_train=X_train_c,
    y_train=y_train_c,
    n_iterations=12,
    cv=3,
    plotly=True,
    export_png=True,
    file_name='bayesian_opt_demo'
)

## 3. Bayesian Neural Networks & Monte Carlo Dropout (`idsw.modelling.deep`)

In [ ]:
# Standard Bayesian Dense Neural Network (Monte Carlo Dropout)
bnn_model, bnn_history = get_bayesian_tf_model(
    X_train=X_train_r,
    y_train=y_train_r,
    type_of_problem='regression',
    dropout_rate=0.2,
    epochs=10,
    batch_size=32
)

# Predict with epistemic uncertainty intervals
mc_mean, mc_std, mc_lower, mc_upper = predict_with_uncertainty(
    model=bnn_model,
    X=X_test_r,
    n_mc_samples=30
)
print(f"MC Dropout Mean[0]: {mc_mean[0]:.2f}, Std[0]: {mc_std[0]:.2f}, 95% CI: [{mc_lower[0]:.2f}, {mc_upper[0]:.2f}]")